In [58]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split


In [2]:
data = pd.read_csv(r"C:\Users\Rohit\Desktop\medAI\dataset\Symptom2Disease.csv")


In [3]:
data

,Unnamed: 0,label,text
0,0,Psoriasis,I have been experiencing a skin rash on my arm...
1,1,Psoriasis,"My skin has been peeling, especially on my kne..."
2,2,Psoriasis,I have been experiencing joint pain in my fing...
3,3,Psoriasis,"There is a silver like dusting on my skin, esp..."
4,4,Psoriasis,"My nails have small dents or pits in them, and..."
...,...,...,...
1195,295,diabetes,I'm shaking and trembling all over. I've lost ...
1196,296,diabetes,"Particularly in the crevices of my skin, I hav..."
1197,297,diabetes,I regularly experience these intense urges and...
1198,298,diabetes,"I have trouble breathing, especially outside. ..."


In [7]:
data.describe()

,Unnamed: 0
count,1200.000000
mean,149.500000
std,86.638166
min,0.000000
25%,74.750000
50%,149.500000
75%,224.250000
max,299.000000


In [8]:
data.tail()

,Unnamed: 0,label,text
1195,295,diabetes,I'm shaking and trembling all over. I've lost ...
1196,296,diabetes,"Particularly in the crevices of my skin, I hav..."
1197,297,diabetes,I regularly experience these intense urges and...
1198,298,diabetes,"I have trouble breathing, especially outside. ..."
1199,299,diabetes,I constantly sneeze and have a dry cough. My i...


Text normalisation function

In [17]:
import re

def normalize_text(text):
    text = text.lower()
    
    # remove special characters
    text = re.sub(r'[^a-z\s]', '', text)
    
    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    # remove stopwords properly (word-based)
    stop_words = ["i", "have", "am", "feel", "since", "for", 
                  "been", "having", "from", "the", "an"]
    
    words = text.split()
    words = [w for w in words if w not in stop_words]
    text = " ".join(words)
    
    # synonym replacement
    replacements = {
        "stomach ache": "abdominal pain",
        "tummy pain": "abdominal pain",
        "high temperature": "fever",
        "breathlessness": "breathing difficulty",
        "head ache": "headache",
        "vomitting": "vomiting"
    }
    
    for key, value in replacements.items():
        text = text.replace(key, value)
    
    # normalize severity
    text = text.replace("high fever", "fever")
    text = text.replace("mild fever", "fever")
    text = text.replace("severe headache", "headache")
    
    return text

In [18]:
data["newText"]= data["text"].apply(normalize_text)

In [19]:
data.head()


,Unnamed: 0,label,text,newText
0,0,Psoriasis,I have been experiencing a skin rash on my arm...,experiencing a skin rash on my arms legs and t...
1,1,Psoriasis,"My skin has been peeling, especially on my kne...",my skin has peeling especially on my knees elb...
2,2,Psoriasis,I have been experiencing joint pain in my fing...,experiencing joint pain in my fingers wrists a...
3,3,Psoriasis,"There is a silver like dusting on my skin, esp...",there is a silver like dusting on my skin espe...
4,4,Psoriasis,"My nails have small dents or pits in them, and...",my nails small dents or pits in them and they ...


In [22]:
all_text = " ".join(data['newText'])
words = all_text.split()

In [23]:
from collections import Counter

word_counts = Counter(words)

In [28]:
print(word_counts.most_common(500))

[('and', 2609), ('my', 1859), ('a', 1266), ('ive', 669), ('of', 533), ('to', 518), ('is', 505), ('in', 435), ('has', 432), ('also', 418), ('skin', 382), ('lot', 353), ('are', 349), ('really', 331), ('pain', 320), ('fever', 272), ('it', 271), ('on', 269), ('im', 258), ('with', 244), ('that', 224), ('me', 224), ('feeling', 217), ('experiencing', 208), ('neck', 178), ('there', 155), ('all', 155), ('had', 152), ('its', 148), ('cough', 148), ('when', 147), ('chest', 147), ('get', 147), ('as', 145), ('quite', 139), ('headache', 134), ('up', 131), ('throat', 131), ('rash', 129), ('hurts', 129), ('red', 114), ('body', 109), ('some', 106), ('severe', 105), ('itching', 105), ('or', 101), ('this', 100), ('over', 99), ('back', 98), ('painful', 98), ('weak', 98), ('chills', 97), ('very', 96), ('difficult', 94), ('discomfort', 89), ('lost', 89), ('nausea', 88), ('like', 87), ('time', 87), ('frequently', 86), ('coughing', 86), ('uncomfortable', 85), ('muscles', 84), ('recently', 80), ('go', 79), ('oc

In [29]:
important_words = [
    "pain", "fever", "cough", "headache", "nausea", "vomiting",
    "dizziness", "fatigue", "chills", "sweating", "weakness",
    
    "chest", "breathing", "difficulty", "blood",
    
    "throat", "abdominal", "stomach", "back", "neck",
    "joints", "muscles", "eyes", "nose", "mouth",
    
    "rash", "itching", "swollen", "red", "pimples", "spots",
    
    "urine", "diarrhea", "constipation", "appetite", "indigestion",
    
    "cold", "sore", "burning", "cramps", "swelling", "phlegm", "numbness"
]

In [30]:
def keep_medical_terms(text):
    words = text.split()
    filtered = [w for w in words if w in important_words]
    return " ".join(filtered)

In [35]:
data['newText'] = data['newText'].apply(keep_medical_terms)

In [36]:
data.head()

,Unnamed: 0,label,text,newText
0,0,Psoriasis,I have been experiencing a skin rash on my arm...,rash red
1,1,Psoriasis,"My skin has been peeling, especially on my kne...",burning
2,2,Psoriasis,I have been experiencing joint pain in my fing...,pain pain joints
3,3,Psoriasis,"There is a silver like dusting on my skin, esp...",back
4,4,Psoriasis,"My nails have small dents or pits in them, and...",


In [42]:
data.loc[3, "text"]

'There is a silver like dusting on my skin, especially on my lower back and scalp. This dusting is made up of small scales that flake off easily when I scratch them.'

In [47]:
data.head()

,Unnamed: 0,label,text,newText
0,0,Psoriasis,I have been experiencing a skin rash on my arm...,rash red
1,1,Psoriasis,"My skin has been peeling, especially on my kne...",burning
2,2,Psoriasis,I have been experiencing joint pain in my fing...,pain pain joints
3,3,Psoriasis,"There is a silver like dusting on my skin, esp...",back
4,4,Psoriasis,"My nails have small dents or pits in them, and...",


In [65]:
# Recreate X
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(data['newText'])

# Recreate y (IMPORTANT)
y = data['label']

# Split again
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [67]:

print(type(data["label"]))

<class 'pandas.core.series.Series'>


In [68]:

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [69]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.7208333333333333


In [85]:
import numpy as np

def predict_disease(text):
    text = normalize_text(text)
    text = merge_phrases(text)
    text = keep_medical_terms(text)
    
    vec = vectorizer.transform([text])
    
    probs = model.predict_proba(vec)[0]
    
    # sort probabilities
    sorted_idx = np.argsort(probs)[::-1]
    
    top1 = sorted_idx[0]
    top2 = sorted_idx[1]
    
    # confidence difference
    diff = probs[top1] - probs[top2]
    
    # threshold decision
    if diff > 0.2:
        return [model.classes_[top1]]   # only 1 disease
    else:
        return [model.classes_[top1], model.classes_[top2]]  # return 2

In [86]:
# def get_urgency(text):
#     if any(x in text for x in ["chestpain", "breathingdifficulty", "unconscious"]):
#         return "HIGH"
    
#     elif any(x in text for x in ["fever", "vomiting", "headache"]):
#         return "MEDIUM"
    
#     else:
#         return "LOW"

In [87]:
def get_urgency(text):
    # HIGH RISK
    if any(x in text for x in [
        "chest pain", "chestpain",
        "breathing difficulty", "breathingdifficulty",
        "difficulty breathing",
        "unconscious", "blood"
    ]):
        return "HIGH"
    
    # MEDIUM
    elif any(x in text for x in [
        "fever", "vomiting", "headache"
    ]):
        return "MEDIUM"
    
    else:
        return "LOW"

In [88]:
def get_advice(urgency):
    if urgency == "HIGH":
        return "Go to hospital immediately"
    elif urgency == "MEDIUM":
        return "Consult a doctor soon"
    else:
        return "Rest and monitor symptoms"

In [89]:
def triage_system(user_input):
    clean = normalize_text(user_input)
    clean = merge_phrases(clean)
    clean = keep_medical_terms(clean)
    
    diseases = predict_disease(user_input)
    urgency = get_urgency(clean)
    advice = get_advice(urgency)
    
    return {
        "symptoms": clean,
        "diseases": diseases,
        "urgency": urgency,
        "advice": advice
    }

In [90]:
triage_system("I have chest pain and difficulty breathing")

{'symptoms': 'chest pain difficulty breathing',
 'diseases': ['drug reaction', 'Pneumonia'],
 'urgency': 'HIGH',
 'advice': 'Go to hospital immediately'}

In [91]:
triage_system("vomiting blood and weakness")

{'symptoms': 'vomiting blood weakness',
 'diseases': ['urinary tract infection', 'Dimorphic Hemorrhoids'],
 'urgency': 'HIGH',
 'advice': 'Go to hospital immediately'}

In [92]:
triage_system("vomiting and nausea for two days")

{'symptoms': 'vomiting nausea',
 'diseases': ['Malaria'],
 'urgency': 'MEDIUM',
 'advice': 'Consult a doctor soon'}

In [93]:
triage_system("I have mild cough and cold")


{'symptoms': 'cough cold',
 'diseases': ['Common Cold', 'Pneumonia'],
 'urgency': 'LOW',
 'advice': 'Rest and monitor symptoms'}